# Lab Day 1 — Notebook Thực Hành

> Notebook đã được điền sẵn code. Bạn chỉ cần vào menu phía trên và chọn **Run All** để thực thi toàn bộ các bước từ việc tải dữ liệu, test mô hình, train baseline, so sánh thử nghiệm cho đến xuất kết quả ra Excel.

In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = ".."     # thư mục gốc repo
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Thiết bị đang dùng: {device}")
if device == "cuda":
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

PyTorch Version: 2.6.0+cu124
Thiết bị đang dùng: cuda
Tên GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Part 0 — Dữ liệu

In [2]:
print("\n--- Chuẩn bị và Nạp Dữ liệu ---")
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

X_tr = data["X_tr"]
mean_train = X_tr[:, :10].mean(dim=0).cpu().numpy()
std_train = X_tr[:, :10].std(dim=0).cpu().numpy()
print(f"Trung bình 10 cột đầu sau khi chuẩn hoá (mong đợi gần 0):\n{np.round(mean_train, 4)}")
print(f"\nĐộ lệch chuẩn 10 cột đầu sau khi chuẩn hoá (mong đợi gần 1):\n{np.round(std_train, 4)}")


--- Chuẩn bị và Nạp Dữ liệu ---
Kích thước X_tr: 371847, X_val: 92962, X_eval: 116203
Accuracy chiến lược đoán lớp đa số trên val: 0.4876
Trung bình 10 cột đầu sau khi chuẩn hoá (mong đợi gần 0):
[ 1.e-04 -0.e+00  0.e+00 -3.e-04 -0.e+00 -0.e+00  0.e+00 -0.e+00  0.e+00
 -0.e+00]

Độ lệch chuẩn 10 cột đầu sau khi chuẩn hoá (mong đợi gần 1):
[1.0001 1.0001 1.0001 1.     1.     1.     0.9999 1.     0.9999 1.0001]


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu

In [3]:
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256,128)]
print(f"Số tham số mô hình: {count_params(model)} (Chuẩn M-base)")

dummy_x = torch.randn(8, 54, device=device)
logits = model(dummy_x)
assert logits.shape == (8, 7)
print(f"Shape Output đã đúng: {logits.shape}")

loss_step0 = evaluate(model, data["X_val"], data["y_val"], "ce")["loss"]
print(f"Loss bước 0 trên Val: {loss_step0:.4f} (Kỳ vọng gần ln(7) = 1.9459)")

print("\nKiểm tra khả năng học (Overfit 20 mẫu)...")
opt = torch.optim.SGD(model.parameters(), lr=0.1)
x_small, y_small = data["X_tr"][:20], data["y_tr"][:20]
for step in range(200):
    model.train()
    opt.zero_grad()
    loss = torch.nn.functional.cross_entropy(model(x_small), y_small)
    loss.backward()
    opt.step()
print(f"Loss sau 200 buớc trên 20 mẫu nhỏ: {loss.item():.6f}")

print("\nKiểm tra gradient đang chảy:")
for name, p in model.named_parameters():
    if p.grad is not None:
        print(f"  {name}: {p.grad.norm().item():.4f}")

Số tham số mô hình: 47879 (Chuẩn M-base)
Shape Output đã đúng: torch.Size([8, 7])
Loss bước 0 trên Val: 1.9514 (Kỳ vọng gần ln(7) = 1.9459)

Kiểm tra khả năng học (Overfit 20 mẫu)...
Loss sau 200 buớc trên 20 mẫu nhỏ: 0.007805

Kiểm tra gradient đang chảy:
  net.0.weight: 0.0111
  net.0.bias: 0.0030
  net.2.weight: 0.0181
  net.2.bias: 0.0018
  net.4.weight: 0.0143
  net.4.bias: 0.0004


## Part 2 — Pipeline và Baseline

In [4]:
cfg_base = {**DEFAULT_CFG, "lr": 0.1, "batch": 512, "epochs": 20}
results_base = []

for s in [1, 2]:
    print(f"\n>>> Đang chạy Baseline (Seed {s}) <<<")
    cfg = {**cfg_base, "seed": s, "exp_id": f"base-s{s}"}
    res = run_experiment(cfg, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{cfg['exp_id']}.png")
    results_base.append(res)
    print(f"-> Kết quả Seed {s}: Best Val Macro-F1 = {res['summary']['val_macro_f1']:.4f}")

f1_scores = [r['summary']['val_macro_f1'] for r in results_base]
print(f"\n=> Độ nhiễu (Noise) qua 2 seed của Baseline Val Macro-F1: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")


>>> Đang chạy Baseline (Seed 1) <<<
-> Kết quả Seed 1: Best Val Macro-F1 = 0.8411

>>> Đang chạy Baseline (Seed 2) <<<
-> Kết quả Seed 2: Best Val Macro-F1 = 0.8472

=> Độ nhiễu (Noise) qua 2 seed của Baseline Val Macro-F1: 0.8442 ± 0.0031


## Part 3 — Thử nghiệm với Adam Optimizer

In [5]:
cfg_adam = {
    **DEFAULT_CFG, 
    "exp_id": "opt-adam-lr1e-3", 
    "group": "optimizer", 
    "description": "Adam optimizer lr=1e-3",
    "optimizer": "adam",
    "lr": 1e-3,
    "epochs": 20
}

print(f"\n>>> Đang chạy Thử nghiệm: {cfg_adam['description']} <<<")
res_adam = run_experiment(cfg_adam, data)
save_result(res_adam, f"{OUT_DIR}/results")
plot_run(res_adam, f"{OUT_DIR}/figures/{cfg_adam['exp_id']}.png")

print(f"-> Kết quả Adam: Best Val Macro-F1 = {res_adam['summary']['val_macro_f1']:.4f}")


>>> Đang chạy Thử nghiệm: Adam optimizer lr=1e-3 <<<
-> Kết quả Adam: Best Val Macro-F1 = 0.8512


In [6]:
# Vẽ ảnh chồng lên để so sánh SGD (Baseline) và Adam
all_results = [results_base[0], res_adam]
plot_compare(all_results, "val_loss", f"{OUT_DIR}/figures/compare_optimizer_val_loss.png", title="Val Loss: SGD vs Adam")
plot_compare(all_results, "val_macro_f1", f"{OUT_DIR}/figures/compare_optimizer_macro_f1.png", title="Val Macro-F1: SGD vs Adam")
print("\nĐã lưu 2 biểu đồ so sánh ở thư mục /figures.")


Đã lưu 2 biểu đồ so sánh ở thư mục /figures.


## Part 4 — Đánh giá trên tập Eval và Ghi Report

In [7]:
# Ta lấy model Adam làm model tốt nhất để dự đoán chấm điểm trên tập Test (Eval)
print("\n--- Đánh giá cấu hình tốt nhất trên tập Eval ---")
final_eval(cfg_adam, res_adam, data, f"{OUT_DIR}/predictions_eval.csv")

print("\nĐang chạy hệ thống chấm điểm (evaluate.py)...")
try:
    result = subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/evaluate.py", "--pred", f"{OUT_DIR}/predictions_eval.csv", "--out", f"{OUT_DIR}/eval_result.json"], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
except Exception as e:
    print(f"Lỗi khi chấm điểm: {e}")


--- Đánh giá cấu hình tốt nhất trên tập Eval ---
Predictions saved to ../predictions_eval.csv

Đang chạy hệ thống chấm điểm (evaluate.py)...

Traceback (most recent call last):
  File "d:\Giselle_\VinAI\K4-Track4-Day1-Neuralnetwork\scripts\evaluate.py", line 121, in <module>
    main()
  File "d:\Giselle_\VinAI\K4-Track4-Day1-Neuralnetwork\scripts\evaluate.py", line 67, in main
    truth = load_eval_labels(args.data, args.meta)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Giselle_\VinAI\K4-Track4-Day1-Neuralnetwork\scripts\evaluate.py", line 37, in load_eval_labels
    meta = pd.read_csv(meta_path)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\nghoo\AppData\Local\anaconda3\Lib\site-packages\pandas\io\parsers\readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\nghoo\AppData\Local\anaconda3\Lib\site-packages\pandas\io\parsers\readers.py", line 620, in _read
    parser = TextFileR

In [8]:
print("\n--- Xuất bảng tổng hợp dữ liệu thí nghiệm Excel ---")
try:
    results_list = load_results(f"{OUT_DIR}/results")
    rows = []
    eval_data = None
    if os.path.exists(f"{OUT_DIR}/eval_result.json"):
        with open(f"{OUT_DIR}/eval_result.json", "r") as f:
            eval_data = json.load(f)
            
    for r in results_list:
        # Điền thêm điểm trên Eval dành riêng cho model được chọn
        if r["cfg"]["exp_id"] == cfg_adam["exp_id"]:
            rows.append(to_row(r, eval_scores=eval_data))
        else:
            rows.append(to_row(r))
            
    write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
    print(f"Thành công! File {OUT_DIR}/experiments.xlsx đã sẵn sàng.")
except Exception as e:
    print(f"Có lỗi khi điền Excel: {e}")


--- Xuất bảng tổng hợp dữ liệu thí nghiệm Excel ---
Thành công! File ../experiments.xlsx đã sẵn sàng.
